In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/fhd-fsl/GenAI-A2
%cd GenAI-A2

Cloning into 'GenAI-A2'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 128 (delta 47), reused 110 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 2.53 MiB | 7.28 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/GenAI-A2


In [3]:
!pip install -U torchao>=0.16.0

In [4]:
!python data/download_dataset.py

README.md: 100% 893/893 [00:00<00:00, 2.34MB/s]

data/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/176k [00:00<?, ?B/s]
data/train-00000-of-00001.parquet: downloading bytes: 100% 157k/157k [00:00<00:00, 296kB/s, 15.4kB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 176k/176k [00:00<00:00, 331kB/s, 17.3kB/s  ]
Generating train split: 100% 1015/1015 [00:00<00:00, 36195.61 examples/s]
Saving to /content/GenAI-A2/data/raw...
Saving the dataset (1/1 shards): 100% 1015/1015 [00:00<00:00, 127798.35 examples/s]
Done.
  Train samples: 1015
  Features: ['annotation_id', 'patient_id', 'note', 'trial_id', 'trial_title', 'criterion_type', 'criterion_text', 'gpt4_explanation', 'explanation_correctness', 'gpt4_sentences', 'expert_sentences', 'gpt4_eligibility', 'expert_eligibility', 'training']


In [5]:
!python data/preprocess.py

Loading raw dataset...
Removing unused columns...
Creating patient-grouped split...
Applying split...
Filter: 100% 1015/1015 [00:00<00:00, 56716.78 examples/s]
Filter: 100% 1015/1015 [00:00<00:00, 49578.64 examples/s]
Filter: 100% 1015/1015 [00:00<00:00, 45413.24 examples/s]
Building split manifest...
Saving...
Split manifest saved to /content/GenAI-A2/data/processed/split_manifest.json
Saving the dataset (1/1 shards): 100% 603/603 [00:00<00:00, 30497.96 examples/s]
Saving the dataset (1/1 shards): 100% 206/206 [00:00<00:00, 18981.67 examples/s]
Saving the dataset (1/1 shards): 100% 206/206 [00:00<00:00, 22371.94 examples/s]
Processed dataset saved to /content/GenAI-A2/data/processed
  train: 603 annotations
  validation: 206 annotations
  test: 206 annotations


In [6]:
!python task1/preprocessing/tokenizer.py

config.json: 100% 661/661 [00:00<00:00, 3.77MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 27.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 74.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 115MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 139MB/s]
Train examples: 603

--- Example 0 ---
Patient: sigir-20143
Criterion: NCT01452971_I_43
Truncated: False (removed 0 tokens)
Input length: 335
Label length: 335
Target JSON:
{"criterion_id": "NCT01452971_I_43", "criterion_type": "inclusion", "criterion_text": "Participant is a lung cancer patient.", "prediction": "not enough information", "evidence_sentence_ids": [0], "evidence_text": ["A 58-year-old nonsmoker white female with mild exertional dyspnea and occasional cough is found to have a left lung mass on chest x-ray."], "explanation": ""}
...

--- Example 1 ---
Patient: sigir-20143
Criterion: NCT01452971_E_44
Truncated: False (removed 0 tokens)
Input length: 338
Label length: 338
Target JSON:
{"criterion_id": "NC

In [9]:
!python task2/eval_task2.py --adapter-path /content/GenAI-A2/task1/outputs/adapter --split test --output-dir /content/drive/MyDrive/GenAI-A2/task2/outputs

Running Task 2 evaluation on test split...
Loading model on cuda...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:17<00:00, 24.13it/s]
Saved 206 predictions to /content/drive/MyDrive/GenAI-A2/task2/outputs/test_predictions.json

--- Computing Metrics ---
Valid predictions: 206/206
Accuracy: 0.7330
Macro F1: 0.5402

Per-class metrics:
  included: P=0.947 R=0.667 F1=0.783 (n=27)
  not included: P=0.000 R=0.000 F1=0.000 (n=7)
  not excluded: P=0.736 R=0.978 F1=0.840 (n=91)
  excluded: P=0.000 R=0.000 F1=0.000 (n=0)
  not enough information: P=0.661 R=0.591 F1=0.624 (n=66)
  not applicable: P=0.714 R=0.333 F1=0.455 (n=15)

Confusion Matrix:
      incl  not   not   excl  not   not 
incl:  18    0    0    0    9    0
not :   0    0    0    0    7    0
not :   0    0   89    0    1    1
excl:   0    0    0    0    0    0
not :   1    0   25    0   39    1
not :   0    0    7    0    3    5

Evidence Retrieval (micro):
  Precision: 0.8319
  